# Experiment 8: Speech Command Classification with MFCC & LSTM

<a href="https://colab.research.google.com/github/mpatel4287-a11y/DLNN-LAB/blob/main/Exp_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
import os
print(os.path.exists("mini_words.zip"))
print(os.path.getsize("mini_words.zip"))




In [ ]:
import os
import zipfile

zip_path = "mini_words.zip"

# Create a separate extraction folder
extract_path = "mini_words"

os.makedirs(extract_path, exist_ok=True)

# Extract ZIP contents
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP file extracted successfully!")


In [ ]:
import os

# Dataset path
dataset_path = "mini_words"

# Display dataset contents
print(os.listdir(dataset_path))

# Identify class folders
commands = [
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
]

# Display classes
print("Speech Commands:", commands)
print("Number of classes:", len(commands))

In [ ]:
!pip install librosa

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
def extract_features(file_path):
    try:
        audio, sample_rate = librosa.load(file_path, sr=16000)

        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sample_rate,
            n_mfcc=40
        )

        # Convert variable-length MFCC into fixed-length feature vector
        mfcc = np.mean(mfcc.T, axis=0)

        return mfcc

    except Exception as e:
        print("Error:", file_path)
        return None
dataset_path = "mini_words"

features = []

commands = [
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
]

for command in commands:

    folder = os.path.join(dataset_path, command)

    for file in os.listdir(folder):

        if file.endswith(".wav"):

            path = os.path.join(folder, file)

            feature = extract_features(path)

            if feature is not None:
                features.append([feature, command])

print("Total samples:", len(features))


In [ ]:
import random
import os

dataset_path = "mini_words"

features = []

random.seed(42)

commands = [
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
]

samples_per_class = 100

for command in commands:

    folder = os.path.join(dataset_path, command)

    wav_files = [
        f for f in os.listdir(folder)
        if f.endswith(".wav")
    ]

    random.shuffle(wav_files)

    wav_files = wav_files[:samples_per_class]

    for file in wav_files:

        path = os.path.join(folder, file)

        feature = extract_features(path)

        if feature is not None:
            features.append([feature, command])

print("Total Samples:", len(features))


In [ ]:
features_df = pd.DataFrame(
    features,
    columns=["feature", "label"]
)

features_df.head()


In [ ]:
X = np.array(features_df["feature"].tolist())
y = np.array(features_df["label"].tolist())

print("X Shape:", X.shape)
print("y Shape:", y.shape)


In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)
print("Classes:")
print(encoder.classes_)



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)
print(X_train.shape)
print(X_test.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

lstm_model = Sequential([
    LSTM(64, input_shape=(40, 1)),
    Dropout(0.4),
    Dense(32, activation="relu"),
    Dense(len(encoder.classes_), activation="softmax")
])

lstm_model.summary()


In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=12,
    batch_size=32
)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("LSTM Accuracy")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

y_pred = lstm_model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))
